# AnomalyCLIP adversarial robustness on MVTec AD and VisA

Set `DATASET` to `'mvtec'`, `'visa'`, `'both'`, `'mvtec_to_visa'`, or `'visa_to_mvtec'`. Cross-dataset modes optimize one universal perturbation only on the source dataset and report effectiveness only on the destination dataset. This notebook is orchestration-only: all attack, model, dataset, metric, and artifact code is loaded from the cloned adversarial-robustness repository.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE/UPDATE REPOSITORIES =====')
working = Path('/kaggle/working')
repo_root = working / 'adversarial-robustness'
anomalyclip_root = working / 'AnomalyCLIP'

repo_url = 'https://github.com/Parsagh05/adversarial-robustness.git'
if repo_root.exists():
    subprocess.run(['git', '-C', str(repo_root), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(repo_root)], check=True)

official_url = 'https://github.com/zqhang/AnomalyCLIP.git'
if anomalyclip_root.exists():
    subprocess.run(['git', '-C', str(anomalyclip_root), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', official_url, str(anomalyclip_root)], check=True)

experiment_root = repo_root
requirements = experiment_root / 'requirements.txt'
print('===== STEP 2: INSTALL DEPENDENCIES =====')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print(f'Experiment code: {experiment_root}')
print(f'Official AnomalyCLIP: {anomalyclip_root}')
print('Environment ready.')

In [ ]:
import os
import sys
from pathlib import Path
import torch

experiment_root = Path('/kaggle/working/adversarial-robustness')
if str(experiment_root) not in sys.path:
    sys.path.insert(0, str(experiment_root))

from adversarial_harness import AttackConfig, ExperimentConfig, run_experiment

print('===== STEP 3: RESOLVE DATA AND CHECKPOINT =====')
# Choose: 'mvtec', 'visa', 'both', 'mvtec_to_visa', or 'visa_to_mvtec'.
DATASET = 'mvtec'
MVTEC_ROOT = Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection')
VISA_ROOT = Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922')
ANOMALYCLIP_ROOT = Path('/kaggle/working/AnomalyCLIP')

VALID_DATASETS = {
    'mvtec', 'visa', 'both', 'mvtec_to_visa', 'visa_to_mvtec'
}
if DATASET not in VALID_DATASETS:
    raise ValueError(f'DATASET must be one of {sorted(VALID_DATASETS)}')
IS_CROSS_DATASET = DATASET in {'mvtec_to_visa', 'visa_to_mvtec'}
EVALUATION_DATASET = {
    'mvtec_to_visa': 'visa',
    'visa_to_mvtec': 'mvtec',
}.get(DATASET, DATASET)

# Use the opposite-dataset checkpoint for zero-shot single-dataset runs.
# A combined run necessarily uses one shared model; by default it uses the
# VisA-trained checkpoint. Change BOTH_CHECKPOINT_DIR if desired.
BOTH_CHECKPOINT_DIR = '9_12_4_multiscale'
checkpoint_dir = {
    'mvtec': '9_12_4_multiscale',       # trained on VisA
    'visa': '9_12_4_multiscale_visa',  # trained on MVTec
    'both': BOTH_CHECKPOINT_DIR,
    'mvtec_to_visa': '9_12_4_multiscale_visa',
    'visa_to_mvtec': '9_12_4_multiscale',
}[DATASET]
CHECKPOINT_PATH = (
    ANOMALYCLIP_ROOT
    / 'checkpoints'
    / checkpoint_dir
    / 'epoch_15.pth'
)
if not CHECKPOINT_PATH.is_file():
    available = sorted((ANOMALYCLIP_ROOT / 'checkpoints').rglob('*.pth'))
    raise FileNotFoundError(
        f'Official checkpoint was not found: {CHECKPOINT_PATH}. '
        f'Available official checkpoints: {available}'
    )

if DATASET in {'mvtec', 'both', 'mvtec_to_visa', 'visa_to_mvtec'} and not MVTEC_ROOT.is_dir():
    raise FileNotFoundError(f'MVTec mount not found: {MVTEC_ROOT}')
if DATASET in {'visa', 'both', 'mvtec_to_visa', 'visa_to_mvtec'} and not VISA_ROOT.is_dir():
    raise FileNotFoundError(f'VisA mount not found: {VISA_ROOT}')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before running this experiment.')

os.environ['ANOMALYCLIP_ROOT'] = str(ANOMALYCLIP_ROOT)
os.environ['ANOMALYCLIP_CHECKPOINT'] = str(CHECKPOINT_PATH)
os.environ['ANOMALYCLIP_CLIP_CACHE'] = '/kaggle/working/clip_cache'

# False gives a small end-to-end validation. Change to True for the full
# Standard modes run 18 conditions. Cross-dataset modes use only the
# transferable dataset-universal scope and therefore run 6 conditions.
FULL_RUN = True
OUTPUT_ROOT = Path(
    f'/kaggle/working/anomalyclip_{DATASET}_adversarial_held_out_full'
    if FULL_RUN else f'/kaggle/working/anomalyclip_{DATASET}_adversarial_held_out_smoke'
)
# Optional reusable artifact produced by kaggle_calibrate_anomalyclip_thresholds.ipynb.
# Leave as None to calibrate once from the selected dataset's normal training split.
# THRESHOLDS_PATH = None
THRESHOLDS_PATH = (
    Path('/kaggle/input/datasets/parsagholami/thresholds/category_thresholds.json')
    if EVALUATION_DATASET == 'mvtec' else None
)

# Reusable matched split generated for the same DATASET collection by
# kaggle_generate_dataset_matched_splits.ipynb.
# Cross-dataset modes use the manifest generated with DATASET='both'.
# Upload that notebook's splits/ output as a Kaggle Dataset, then update
# SPLIT_MANIFEST_ROOT to its mounted location.
USE_SPLIT_MANIFEST = True
SPLIT_MANIFEST_ROOT = Path(
    '/kaggle/input/datasets/parsagholami/mvtec-matched-splits/splits'
)
MANIFEST_DATASET = 'both' if IS_CROSS_DATASET else DATASET
split_artifact_name = f'{MANIFEST_DATASET}_matched_test_per_category_v1_seed111'
SPLIT_MANIFEST_CSV = SPLIT_MANIFEST_ROOT / f'{split_artifact_name}.csv'
SPLIT_MANIFEST_JSON = SPLIT_MANIFEST_ROOT / f'{split_artifact_name}.json'
if USE_SPLIT_MANIFEST:
    for manifest_path in (SPLIT_MANIFEST_CSV, SPLIT_MANIFEST_JSON):
        if not manifest_path.is_file():
            raise FileNotFoundError(f'Split manifest not found: {manifest_path}')


if FULL_RUN:
    categories = None
    scopes = (('dataset',) if IS_CROSS_DATASET else ('per_image', 'per_category', 'dataset'))
    directions = ('normal_to_abnormal', 'abnormal_to_normal')
    loss_modes = ('global', 'local', 'combined')
    steps = 20
    universal_steps = 200
    max_samples = None
    compute_lpips = True
else:
    categories = {
        'mvtec': ('bottle',),
        'visa': ('candle',),
        'both': ('bottle', 'candle'),
        'mvtec_to_visa': ('candle',),
        'visa_to_mvtec': ('bottle',),
    }[DATASET]
    source_categories = {
        'mvtec_to_visa': ('bottle',),
        'visa_to_mvtec': ('candle',),
    }.get(DATASET)
    scopes = (('dataset',) if IS_CROSS_DATASET else ('per_image',))
    directions = ('abnormal_to_normal',)
    loss_modes = ('combined',)
    steps = 2
    universal_steps = 2
    max_samples = 4
    compute_lpips = False

if FULL_RUN:
    source_categories = None

attack = AttackConfig(
    image_size=518,
    epsilon=8 / 255,
    step_size=2 / 255,
    steps=steps,
    universal_steps=universal_steps,
    random_start=True,
    temperature=0.07,
    global_weight=0.2,
    local_weight=0.8,
    feature_layers=(6, 12, 18, 24),
    scopes=scopes,
    directions=directions,
    loss_modes=loss_modes,
    per_image_batch_size=1,
    universal_batch_size=2,
    seed=111,
)

config = ExperimentConfig(
    dataset=DATASET,
    mvtec_root=str(MVTEC_ROOT),
    visa_root=str(VISA_ROOT),
    output_root=str(OUTPUT_ROOT),
    anomalyclip_root=str(ANOMALYCLIP_ROOT),
    anomalyclip_checkpoint=str(CHECKPOINT_PATH),
    device='cuda',
    target_model='AnomalyCLIP',
    categories=categories,
    source_categories=source_categories,
    target_batch_size=2,
    metric_size=518,
    aupro_fpr_limit=0.30,
    aupro_max_thresholds=200,
    anomaly_map_sigma=4.0,
    compute_lpips=compute_lpips,
    save_universal_perturbations=True,
    save_adversarial_examples=5 if FULL_RUN else 2,
    universal_protocol='held_out',
    fit_fraction=0.5,
    split_seed=111,
    diagnostic_max_samples=64,
    threshold_mode='normal_train_quantile',
    threshold_quantile=0.95,
    thresholds_path=str(THRESHOLDS_PATH) if THRESHOLDS_PATH else None,
    max_samples_per_category=max_samples,
    use_split_manifest=USE_SPLIT_MANIFEST,
    split_manifest_csv=(
        str(SPLIT_MANIFEST_CSV) if USE_SPLIT_MANIFEST else None
    ),
    split_manifest_json=(
        str(SPLIT_MANIFEST_JSON) if USE_SPLIT_MANIFEST else None
    ),
    resume=True,
    attack=attack,
    target_kwargs={'clip_download_root': '/kaggle/working/clip_cache'},
)

print('===== STEP 4: RUN EXPERIMENT =====')
print('Mode:', 'FULL' if FULL_RUN else 'SMOKE')
print('Dataset:', DATASET)
print('Checkpoint:', CHECKPOINT_PATH)
print('Output:', OUTPUT_ROOT)
summary_path = run_experiment(config)
print('Finished. Summary:', summary_path)

In [ ]:
import shutil
from pathlib import Path

print('===== STEP 5: PACKAGE OUTPUTS =====')
output_root = OUTPUT_ROOT
if not output_root.is_dir():
    raise FileNotFoundError(f'Output directory was not created: {output_root}')
archive = shutil.make_archive(
    str(output_root),
    'zip',
    root_dir=output_root.parent,
    base_dir=output_root.name,
)
print('Packaged results:', archive)